# 05. Train/Test Split and Data Leakage

**Before this notebook:** read `notes/05_train_test_split.md`.

**Goal:** split the cleaned data into a training set and a test set the right way, check the split is fair, see data leakage with your own eyes, and save the four pieces for the next notebooks.

**Steps**
1. Load the cleaned data and separate X and y
2. Practice on a tiny example
3. Split with and without `stratify`
4. See data leakage in action
5. Save the split
6. Exercise

## Step 1: Load the cleaned data and separate X and y

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../../data/processed/telco_clean.csv")

# Target: turn "Yes"/"No" into 1/0. Models need numbers. 1 = churned (the class we care about).
y = (df["Churn"] == "Yes").astype(int)

# Features: drop the target and the ID column.
X = df.drop(columns=["Churn", "customerID"])

print("X:", X.shape, "| y:", y.shape)
print("Churn rate overall:", round(y.mean() * 100, 1), "%")

X: (7043, 19) | y: (7043,)
Churn rate overall: 26.5 %


**Line by line**
- `from sklearn.model_selection import train_test_split`: scikit-learn's splitting function.
- `StandardScaler` is imported now because we use it in Step 4 to demonstrate leakage.
- `(df["Churn"] == "Yes").astype(int)`: True/False becomes 1/0. We call 1 the **positive class** (the event we want to detect). This choice matters later, because metrics such as precision and recall are about the positive class.
- `X = df.drop(columns=[...])`: all remaining columns are the features. We keep the original text categories, because encoding happens *after* the split (notebook 06).

**Expected output:** `X: (7043, 19) | y: (7043,)` and a churn rate of about 26.5 %.

## Step 2: Practice on a tiny example

The 10 bills from the notes, split 80/20.

In [2]:
bills = pd.DataFrame({
    "bill":  [20, 22, 25, 30, 35, 40, 45, 50, 55, 200],
    "left":  [0,  0,  0,  1,  0,  1,  0,  0,  1,  1],
})

X_small = bills[["bill"]]       # double brackets keep it a table (X must be 2-dimensional)
y_small = bills["left"]

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_small, y_small, test_size=0.2, random_state=42
)

print("Train rows:", list(Xs_train.index))
print("Test rows: ", list(Xs_test.index))

Train rows: [5, 0, 7, 2, 9, 4, 3, 6]
Test rows:  [8, 1]


**Line by line**
- `train_test_split(X, y, test_size=0.2, random_state=42)` *takes* features and target, and *returns four objects* in this fixed order: `X_train, X_test, y_train, y_test`.
- `test_size=0.2`: 20 % of 10 rows = 2 test rows.
- The index labels show which rows were chosen. They look shuffled (not simply the last two), because the function randomises first. **Run the cell again: the same rows appear**, because of `random_state=42`. Change 42 to 7 and different rows are picked.

## Step 3: Split the real data, with and without `stratify`

First, a split **without** stratification, to see what can go wrong. Then the correct split.

In [3]:
# Without stratify
_, _, y_tr_plain, y_te_plain = train_test_split(X, y, test_size=0.2, random_state=1)

# With stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Overall churn rate:         ", round(y.mean() * 100, 2), "%")
print("Not stratified  -> train:", round(y_tr_plain.mean() * 100, 2), "% | test:", round(y_te_plain.mean() * 100, 2), "%")
print("Stratified      -> train:", round(y_train.mean() * 100, 2), "% | test:", round(y_test.mean() * 100, 2), "%")
print()
print("Sizes: train", X_train.shape, "| test", X_test.shape)

Overall churn rate:          26.54 %
Not stratified  -> train: 27.0 % | test: 24.7 %
Stratified      -> train: 26.54 % | test: 26.54 %

Sizes: train (5634, 19) | test (1409, 19)


**Line by line**
- `_` is a Python convention for "I do not need this value". We only want the two target pieces for the comparison.
- `stratify=y`: when cutting, keep the 0/1 proportions of `y` the same in both parts.

**How to read the output**
- Without stratify, the train and test churn rates can drift slightly apart from 26.5 %. With 7,043 rows the drift is small, but it can be large on small or very imbalanced datasets (imagine a 2 % minority class ending up as 0 % of the test set).
- With stratify, both parts match the overall rate almost exactly. So later, a score on the test set is comparable to the training data.
- Sizes: 5,634 training rows and 1,409 test rows (80 / 20).

**From now on we use the stratified split with `random_state=42`.**

In [4]:
# Quick sanity checks: no row in both sets, nothing lost
assert set(X_train.index).isdisjoint(set(X_test.index)), "Overlap between train and test!"
assert len(X_train) + len(X_test) == len(X)
print("Split is clean: no overlap, no missing rows.")

Split is clean: no overlap, no missing rows.


**Why check?** A mistake here would silently ruin everything after it. `isdisjoint` returns True when two sets share no members. These two lines cost nothing and catch bugs early.

## Step 4: See data leakage in action

We will scale `MonthlyCharges` (subtract the mean, divide by the standard deviation) in two ways. `StandardScaler` learns the mean and standard deviation with `fit` and applies them with `transform`.

- **Wrong way:** fit the scaler on **all** the data, then split.
- **Right way:** fit the scaler on the **training set only**, then use it on the test set.

In [5]:
col = ["MonthlyCharges"]

# WRONG: learn from everything (train + test mixed)
leaky_scaler = StandardScaler().fit(X[col])

# RIGHT: learn from training data only
proper_scaler = StandardScaler().fit(X_train[col])

print("Mean learned (WRONG, all data):  ", round(leaky_scaler.mean_[0], 4))
print("Mean learned (RIGHT, train only):", round(proper_scaler.mean_[0], 4))
print()
print("Std learned  (WRONG, all data):  ", round(leaky_scaler.scale_[0], 4))
print("Std learned  (RIGHT, train only):", round(proper_scaler.scale_[0], 4))
print()
print("Mean of the test set itself:     ", round(X_test[col].mean().iloc[0], 4))

Mean learned (WRONG, all data):   64.7617
Mean learned (RIGHT, train only): 64.93

Std learned  (WRONG, all data):   30.0879
Std learned  (RIGHT, train only): 30.1354

Mean of the test set itself:      64.0889


**Line by line**
- `StandardScaler().fit(X[col])`: *learns* the mean (`mean_`) and standard deviation (`scale_`) of the column. *Input:* a table of numbers. *Output:* the fitted scaler. Nothing is changed in the data yet.
- Using `fit` on `X` (everything) is **the leak**: the 1,409 test rows contribute to the numbers.

**How to read the output**
- The two sets of numbers are **different**, though the difference is small here because the split is random and large. The leaky mean is partly built from test rows.
- In real life, **new customers never arrive with their data already averaged in**. The proper scaler imitates that: it knows only the training data.

**Why is the effect small here but it still matters?**
1. The habit matters. With small datasets, rare categories, or steps like feature selection or target-based encoding, leakage can inflate scores a lot.
2. Leakage gets hidden: it is invisible in the code results, and shows up only when the model fails in production.

The correct application:

In [6]:
X_train_scaled = proper_scaler.transform(X_train[col])   # apply to train
X_test_scaled  = proper_scaler.transform(X_test[col])    # apply the SAME numbers to test

print("Train after scaling: mean", round(X_train_scaled.mean(), 3), "| std", round(X_train_scaled.std(), 3))
print("Test  after scaling: mean", round(X_test_scaled.mean(), 3),  "| std", round(X_test_scaled.std(), 3))

Train after scaling: mean -0.0 | std 1.0
Test  after scaling: mean -0.028 | std 0.992


**How to read the output**
- The training data has mean about 0 and standard deviation about 1, **exactly** as designed.
- The test data is **close to** 0 and 1 but not exactly. **That is correct and expected.** The test set was scaled with numbers it did not influence, just like a new customer would be. If the test mean were exactly 0, that would suggest a leak.

This is only a demonstration. We do **not** keep this scaled column. The real scaling happens inside a pipeline in notebooks 06 and 08.

## Step 5: Save the split

So every following notebook uses the **same** train/test rows, we save them.

In [7]:
OUT_DIR = Path("../../data/processed")

X_train.to_csv(OUT_DIR / "X_train.csv", index=False)
X_test.to_csv(OUT_DIR / "X_test.csv", index=False)
y_train.to_csv(OUT_DIR / "y_train.csv", index=False)
y_test.to_csv(OUT_DIR / "y_test.csv", index=False)

print("Saved:", sorted(p.name for p in OUT_DIR.glob("*_*.csv")))

Saved:

 ['X_test.csv', 'X_train.csv', 'telco_clean.csv', 'y_test.csv', 'y_train.csv']


**Line by line**
- `to_csv(..., index=False)`: writes each piece to its own file, without the row-number column.
- `OUT_DIR / "X_train.csv"`: the `/` operator on a `Path` joins folder and file names.
- `OUT_DIR.glob("*_*.csv")`: lists the files matching the pattern, so we can confirm they exist.

**Expected output:** a list containing `X_test.csv, X_train.csv, y_test.csv, y_train.csv` (and `telco_clean.csv`, which also matches the pattern).

**The rule from now on:** the test files are not opened again until the final evaluation. All exploration, scaling choices and model tuning use **only** the training files.

## Summary

| Decision | Choice | Reason |
|---|---|---|
| Split ratio | 80 % train / 20 % test | Standard balance |
| `stratify` | `y` | Keep 26.5 % churn in both parts |
| `random_state` | 42 | Reproducible |
| Target | 1 = churn, 0 = stay | Models need numbers, 1 is the class we want to detect |
| Dropped | `customerID` | An ID has no pattern to learn |

**Next:** `06_encoding_and_scaling`: turn the text categories into numbers and scale the numeric features. We will fit everything on the training set only.

---
## Exercise

1. Change `test_size` to 0.3. How many rows are in each part?
2. Run the split twice with `random_state=42`, then once with `random_state=10`. Compare `X_train.index[:5]` for the three runs. What do you notice?
3. Use `MinMaxScaler` (from `sklearn.preprocessing`) instead of `StandardScaler`: fit it on `X_train[["tenure"]]`, then print the minimum and maximum of the *transformed test* values. Are they exactly 0 and 1? Why or why not?
4. **Think:** a colleague fills missing values with the column median *before* splitting "to save time". Explain in two sentences why this is leakage.

In [8]:
# Exercise 1

In [9]:
# Exercise 2